In [1]:
# import libraries
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp

import ee
import geemap
import rasterio

# import modules
from losses import CombinedLoss
from models import build_unet, MiddleFusionUNet, SegFormer
from dataset import PatchDataset, PairedDataset, train_transform
from train import train_config, train_config_dual
from bootstrap import bootstrap, bootstrap_dual, resampling, compare_models
from preprocessing import extract_and_filter_patches, file_upload, percentile_normalise
from splitting import compute_patch_proportions, compute_block_proportions, stratified_block_split, assign_patch_splits, compute_class_weights
from evaluate import evaluate_macro_f1, evaluate_full_metrics, evaluate_full_metrics_dual, plot_confusion_matrices_for_dissertation, compute_error_map, plot_error_map_comparison

# Environment setup
from google.colab import drive
drive.mount('/content/drive')

ee.Authenticate()
ee.Initialize(project='amenities-488314')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Define the study area
admin_boundaries = ee.FeatureCollection('FAO/GAUL/2015/level2')
study_area = admin_boundaries.filter(
    ee.Filter.inList(
        'ADM2_NAME',
        ['Port Harcourt', 'Obio/Akpor']
    )
)

# Define the study period
start_date = '2024-01-01'
end_date = '2026-04-01'

# Sentinel-1 collection
sentinel1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.calendarRange(12, 3, 'month'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select(['VV', 'VH'])
)


# Sentinel-2 cloud and quality masking function
def sentinel2_cloud_mask(image):
    scl = image.select('SCL')
    clear_mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
    )
    return (
        image
        .select(['B2', 'B3', 'B4', 'B8'])
        .updateMask(clear_mask)
        .divide(10000)
        .copyProperties(image, ['system:time_start'])
    )


# Sentinel-2 collection
sentinel2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.calendarRange(12, 3, 'month'))
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 10))
    .map(sentinel2_cloud_mask)
)


# Function to create a median composite and clip it to the study area
def composite_clip(collection):
    composite = collection.median()
    return composite.clip(study_area)


# Create the Sentinel median composite
sentinel1_composite = composite_clip(sentinel1_collection)
sentinel2_composite = composite_clip(sentinel2_collection)

In [ ]:
class_names = {
    0: 'Dense Vegetation',
    1: 'Sparse Vegetation',
    2: 'Water Bodies',
    3: 'Built-up Area'
}

# Pull Dynamic World label composite using MODE (most frequent class per pixel)
dw_collection = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate(start_date, end_date)
    .filterBounds(study_area)
    .filter(ee.Filter.calendarRange(12, 3, 'month'))
    .select('label')
)

print("Dynamic World images found:", dw_collection.size().getInfo())

dw_composite = dw_collection.mode().clip(study_area.geometry())

# Remap Dynamic World's classes to the 4 target classes
# DW:     0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops, 5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice
# Study:  0=Dense Vegetation, 1=Sparse Vegetation, 2=Water Bodies, 3=Built-up Area
remap_from = [0, 1, 2, 3, 4, 5, 6, 7]
remap_to = [2, 0, 1, 0, 1, 1, 3, 1]

labels = dw_composite.remap(remap_from, remap_to).rename('label')

In [ ]:
def export_image(image, description, folder):
    """
    Exports an image to Google Drive as a GeoTIFF, filling masked
    pixels with a NoData sentinel value.

    Args:
        image (ee.Image): image to export.
        description (str): export task name / output filename.
        folder (str): destination Drive folder.
    """
    image_with_nodata = image.unmask(-9999)
    task = ee.batch.Export.image.toDrive(
        image=image_with_nodata,
        description=description,
        folder=folder,
        scale=10,
        crs='EPSG:32632',
        region=study_area.geometry(),
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'noData': -9999}
    )
    task.start()
    return task

# These calls start the three Earth Engine exports defined above; wait for them to finish in Drive before running file_upload() below.
export_image(sentinel1_composite, 'Sentinel1_Composite', 'Dissertation_PH')
export_image(sentinel2_composite, 'Sentinel2_Composite', 'Dissertation_PH')
export_image(labels, 'DynamicWorld_Labels', 'Dissertation_PH')


In [ ]:
# Load the exported GeoTIFFs from Drive
sentinel1_path = "/content/drive/MyDrive/Dissertation_PH/Sentinel1_Composite.tif"
sentinel2_path = "/content/drive/MyDrive/Dissertation_PH/Sentinel2_Composite.tif"
dynamic_world_path = "/content/drive/MyDrive/Dissertation_PH/DynamicWorld_Labels.tif"

sentinel1, sentinel1_profile = file_upload(sentinel1_path)
sentinel2, sentinel2_profile = file_upload(sentinel2_path)
dynamic_world_label, dw_profile = file_upload(dynamic_world_path)

# Normalise each sensor separately, then fuse into one 6-channel array
sentinel1_normalised = percentile_normalise(sentinel1, ['VV', 'VH'])
sentinel2_normalised = percentile_normalise(sentinel2, ['Blue', 'Green', 'Red', 'NIR'])
sentinel_fused = np.concatenate([sentinel1_normalised, sentinel2_normalised], axis=0)

# Record channel counts, needed later for model construction
n_s1_bands = sentinel1_normalised.shape[0]  # 2 — VV, VH
n_s2_bands = sentinel2_normalised.shape[0]  # 4 — Blue, Green, Red, NIR
n_total_bands = n_s1_bands + n_s2_bands  # total channels once S1 and S2 are fused together

In [ ]:
# Extract fixed-size patches, discarding those with too much missing data
fused_patches, label_patches, positions, stats = extract_and_filter_patches(
    sentinel_fused, dynamic_world_label, patch_size=256, stride=128, nan_threshold=0.5
)

# Split each fused patch back into its Sentinel-1 and Sentinel-2 channels
sentinel1_patches = [p[:n_s1_bands] for p in fused_patches]
sentinel2_patches = [p[n_s1_bands:] for p in fused_patches]

In [ ]:
# Compute per-patch class proportions, group patches into spatial blocks,
# and split blocks into train/val/test using stratified sampling
proportions = compute_patch_proportions(label_patches, num_classes=4)
patches_df, block_props = compute_block_proportions(positions, proportions, block_size=512)
train_blocks, val_blocks, test_blocks = stratified_block_split(block_props, test_size=0.15, val_size=0.15, class_threshold=0.005, random_state=42)
patches_df = assign_patch_splits(patches_df, train_blocks, val_blocks, test_blocks)

# Convert the split column into patch ID arrays for each split
train_ids = patches_df[patches_df['split'] == 'train']['patch_id'].to_numpy()
val_ids = patches_df[patches_df['split'] == 'val']['patch_id'].to_numpy()
test_ids = patches_df[patches_df['split'] == 'test']['patch_id'].to_numpy()

# Compute class weights from the training split only
train_weights = compute_class_weights(patches_df[patches_df['split'] == 'train'], num_classes=4)
weights_tensor = torch.tensor(train_weights, dtype=torch.float32)

In [ ]:
# Fused (6-channel) datasets, for Early Fusion and RQ2 architecture comparison
train_ds_fusion = PatchDataset(fused_patches, label_patches, train_ids, transform=train_transform)
val_ds_fusion = PatchDataset(fused_patches, label_patches, val_ids)
test_ds_fusion = PatchDataset(fused_patches, label_patches, test_ids)

# Single-sensor datasets, using the pre-sliced patch lists
train_ds_s1 = PatchDataset(sentinel1_patches, label_patches, train_ids, transform=train_transform)
val_ds_s1 = PatchDataset(sentinel1_patches, label_patches, val_ids)
test_ds_s1 = PatchDataset(sentinel1_patches, label_patches, test_ids)

train_ds_s2 = PatchDataset(sentinel2_patches, label_patches, train_ids, transform=train_transform)
val_ds_s2 = PatchDataset(sentinel2_patches, label_patches, val_ids)
test_ds_s2 = PatchDataset(sentinel2_patches, label_patches, test_ids)

# Middle Fusion datasets: pairs of single-sensor PatchDatasets, sliced by channel
train_ds_middle = PairedDataset(
    PatchDataset(fused_patches, label_patches, train_ids, channel_slice=slice(0, n_s1_bands), transform=train_transform),
    PatchDataset(fused_patches, label_patches, train_ids, channel_slice=slice(n_s1_bands, n_total_bands), transform=train_transform),
)
val_ds_middle = PairedDataset(
    PatchDataset(fused_patches, label_patches, val_ids, channel_slice=slice(0, n_s1_bands)),
    PatchDataset(fused_patches, label_patches, val_ids, channel_slice=slice(n_s1_bands, n_total_bands)),
)
test_ds_middle = PairedDataset(
    PatchDataset(fused_patches, label_patches, test_ids, channel_slice=slice(0, n_s1_bands)),
    PatchDataset(fused_patches, label_patches, test_ids, channel_slice=slice(n_s1_bands, n_total_bands)),
)

In [ ]:
CHECKPOINT_DIR = '/content/drive/MyDrive/Dissertation_PH/checkpoints'

# S1-only: convolutional baseline, radar channels only
model_s1 = build_unet(encoder_name="resnet34", in_channels=n_s1_bands, n_classes=len(class_names))
model_s1 = model_s1.to(device)

# S2-only: convolutional baseline, optical channels only
model_s2 = build_unet(encoder_name="resnet34", in_channels=n_s2_bands, n_classes=len(class_names))
model_s2 = model_s2.to(device)

# Early Fusion: both sensors combined at input, single shared encoder
model_fusion = build_unet(encoder_name="resnet34", in_channels=n_total_bands, n_classes=len(class_names))
model_fusion = model_fusion.to(device)

# Middle Fusion: custom dual-encoder architecture, sensors combined per encoder stage
model_middle_fusion = MiddleFusionUNet(n_s1_bands=n_s1_bands, n_s2_bands=n_s2_bands, n_classes=len(class_names))
model_middle_fusion = model_middle_fusion.to(device)

In [ ]:
# Train S1-only
ckpt_path_s1, best_f1_s1, history_s1 = train_config(
    model=model_s1,
    config_name='s1_only_res34_L6-4_WD-1e4',
    train_ds=train_ds_s1,
    val_ds=val_ds_s1,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Train S2-only
ckpt_path_s2, best_f1_s2, history_s2 = train_config(
    model=model_s2,
    config_name='s2_only_res34_L6-4_WD-1e4',
    train_ds=train_ds_s2,
    val_ds=val_ds_s2,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Train Early Fusion
ckpt_path_fusion, best_f1_fusion, history_fusion = train_config(
    model=model_fusion,
    config_name='early_fusion_res34_L6-4_WD-1e4',
    train_ds=train_ds_fusion,
    val_ds=val_ds_fusion,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

# Train Middle Fusion (dual-input, uses train_config_dual and the paired datasets)
ckpt_path_middle, best_f1_middle, history_middle = train_config_dual(
    model=model_middle_fusion,
    config_name='middle_fusion_res34_L6-4_WD-1e4',
    train_ds=train_ds_middle,
    val_ds=val_ds_middle,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

In [ ]:
# Load each best checkpoint and evaluate on the held-out test set
test_loader_s1 = DataLoader(test_ds_s1, batch_size=8, shuffle=False)
test_loader_s2 = DataLoader(test_ds_s2, batch_size=8, shuffle=False)
test_loader_fusion = DataLoader(test_ds_fusion, batch_size=8, shuffle=False)
test_loader_middle = DataLoader(test_ds_middle, batch_size=8, shuffle=False)

# S1-only
model_s1.load_state_dict(torch.load(ckpt_path_s1))
results_s1, f1_pc_s1, prec_pc_s1, recall_pc_s1, cm_s1 = evaluate_full_metrics(
    model_s1, test_loader_s1, len(class_names), class_names, device
)

# S2-only
model_s2.load_state_dict(torch.load(ckpt_path_s2))
results_s2, f1_pc_s2, prec_pc_s2, recall_pc_s2, cm_s2 = evaluate_full_metrics(
    model_s2, test_loader_s2, len(class_names), class_names, device
)

# Early Fusion
model_fusion.load_state_dict(torch.load(ckpt_path_fusion))
results_fusion, f1_pc_fusion, prec_pc_fusion, recall_pc_fusion, cm_fusion = evaluate_full_metrics(
    model_fusion, test_loader_fusion, len(class_names), class_names, device
)

# Middle Fusion (dual-input, uses evaluate_full_metrics_dual)
model_middle_fusion.load_state_dict(torch.load(ckpt_path_middle))
results_middle, f1_pc_middle, prec_pc_middle, recall_pc_middle, cm_middle = evaluate_full_metrics_dual(
    model_middle_fusion, test_loader_middle, len(class_names), class_names, device
)

In [ ]:
# Collect predictions across all four RQ1 configurations
preds_s1, labels_s1 = bootstrap(model_s1, test_ds_s1, device)
preds_s2, labels_s2 = bootstrap(model_s2, test_ds_s2, device)
preds_fusion, labels_fusion = bootstrap(model_fusion, test_ds_fusion, device)
preds_middle, labels_middle = bootstrap_dual(model_middle_fusion, test_ds_middle, device)

# Bootstrap resample macro-F1 for each configuration
f1_s1 = resampling(preds_s1, labels_s1)
f1_s2 = resampling(preds_s2, labels_s2)
f1_fusion = resampling(preds_fusion, labels_fusion)
f1_middle = resampling(preds_middle, labels_middle)

# Compare Early Fusion against each alternative
compare_models(f1_fusion, f1_s1, "Early Fusion", "S1-only", lower=2.5, upper=97.5)
compare_models(f1_fusion, f1_s2, "Early Fusion", "S2-only", lower=2.5, upper=97.5)
compare_models(f1_fusion, f1_middle, "Early Fusion", "Middle Fusion", lower=2.5, upper=97.5)

In [ ]:
# SegFormer: transformer encoder, paired with its own native decoder (custom implementation)
model_segformer = SegFormer(encoder_name="mit_b2", in_channels=n_total_bands, n_classes=len(class_names))
model_segformer = model_segformer.to(device)

ckpt_path_segformer, best_f1_segformer, history_segformer = train_config(
    model=model_segformer,
    config_name='segformer_mitb2_L6-4_WD-1e4',
    train_ds=train_ds_fusion,
    val_ds=val_ds_fusion,
    weights_tensor=weights_tensor,
    n_classes=len(class_names),
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    max_epochs=100,
    patience=12,
    batch_size=8,
    lr=1e-4,
    weight_decay=1e-4,
)

In [ ]:
# SegFormer, F1-selected checkpoint
model_segformer.load_state_dict(torch.load(ckpt_path_segformer))
results_segformer, f1_pc_segformer, prec_pc_segformer, recall_pc_segformer, cm_segformer = evaluate_full_metrics(
    model_segformer, test_loader_fusion, len(class_names), class_names, device
)

# Bootstrap test: is SegFormer's numerical lead over U-Net statistically real?
preds_segformer, labels_segformer = bootstrap(model_segformer, test_ds_fusion, device)
f1_segformer = resampling(preds_segformer, labels_segformer)

compare_models(f1_fusion, f1_segformer, "U-Net", "SegFormer", lower=2.5, upper=97.5)

In [ ]:
# confusion matrices for U-Net and SegFormer (Figure 4.2)
plot_confusion_matrices_for_dissertation(
    cm_fusion, cm_segformer, class_names,
    "(a) U-Net (ResNet34)", "(b) SegFormer",
    normalize=True
)

# Ground truth, predictions, and error maps for a representative test patch (Figure 4.3)
idx = 5
img, true_label = test_ds_fusion[idx]

with torch.no_grad():
    pred_unet = model_fusion(img.unsqueeze(0).to(device)).argmax(dim=1).cpu().squeeze().numpy()
    pred_segformer = model_segformer(img.unsqueeze(0).to(device)).argmax(dim=1).cpu().squeeze().numpy()

plot_error_map_comparison(
    true_label.numpy(), pred_unet, pred_segformer, class_names,
    "U-Net", "SegFormer"
)